In [2]:
pip install openai

   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   -------------------------------------- - 1.3/1.4 MB 7.7 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 5.3 MB/s  0:00:00

   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ------------------- 1/2 [openai]
   -------------------- ----------------

In [19]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import time
import pandas as pd
from openai import RateLimitError
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report

In [7]:
# Loading API key through a .env file. The .env file is not uploaded to github.

load_dotenv('the_nlp.env')  
api_key = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

In [21]:
# Importing and checking speeches
# Potential for function here

h5 = pd.read_csv('hansard500.csv')

print(h5.shape, '\n')
print(h5.isnull().sum(), '\n')
print(h5['speech_class'].value_counts(), '\n')
print(h5['party'].value_counts(), '\n')

h5.head()

(500, 8) 

speech            0
party            19
constituency     19
date              0
speech_class      0
major_heading     0
year              0
speakername       0
dtype: int64 

speech_class
Speech        481
Procedural     16
Division        3
Name: count, dtype: int64 

party
Conservative                        313
Labour                               88
Scottish National Party              32
Labour (Co-op)                       19
Speaker                              11
Democratic Unionist Party             7
Liberal Democrat                      5
Plaid Cymru                           3
Independent                           1
Social Democratic & Labour Party      1
Green Party                           1
Name: count, dtype: int64 



,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick
3,In order to allow the safe exit of hon. Member...,Speaker,Chorley,2020-10-07,Speech,Prime Minister,2020,Lindsay Hoyle
4,Welsh lamb is a premium product that is wanted...,Conservative,Montgomeryshire,2021-02-03,Speech,Wales,2021,Craig Williams


In [23]:
# Make a note that I would take out Lib Dems because of such low nr. of examples if I were doing this professionally

h5['party'] = h5['party'].replace('Labour (Co-op)', 'Labour')
h5 = h5[h5['speech_class'].isin(['Speech'])]
h5 = h5[h5['party'].isin(['Conservative', 'Labour', 'Scottish National Party', 'Liberal Democrat'])]

print(h5['speech_class'].value_counts(), '\n')
print(h5['party'].value_counts(), '\n')

h5.head()

speech_class
Speech    457
Name: count, dtype: int64 

party
Conservative               313
Labour                     107
Scottish National Party     32
Liberal Democrat             5
Name: count, dtype: int64 



,speech,party,constituency,date,speech_class,major_heading,year,speakername
0,We will now suspend for three minutes for sani...,Conservative,Ribble Valley,2021-03-11,Speech,Contingencies Fund (No. 2) Bill,2021,Nigel Evans
1,I am now beginning to share the indignation of...,Labour,City of Chester,2020-11-24,Speech,Exiting the European Union,2020,Christian Matheson
2,"The hon. Lady is wrong; as a matter of fact, w...",Conservative,Newark,2021-02-10,Speech,Building Safety,2021,Robert Jenrick
4,Welsh lamb is a premium product that is wanted...,Conservative,Montgomeryshire,2021-02-03,Speech,Wales,2021,Craig Williams
5,"Given that this was outlined earlier today, it...",Conservative,Great Yarmouth,2021-04-13,Speech,Northern Ireland,2021,Brandon Lewis


In [ ]:
x = h5['speech']          
y = h5['party']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, stratify=y, random_state=26)                    # Same seed as part 2 for consistency

labels = sorted(y.unique().tolist())                      # Creating a list of party labels to feed into LLM
label_str = ", ".join(labels)

In [16]:
# function to talk to LLM
# Built in rate limit error catcher

def classify(prompt, max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model="meta-llama/llama-3.3-70b-instruct:free",
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=10,
            )
            return resp.choices[0].message.content.strip()
        except RateLimitError:
            wait = 30 * (attempt + 1)   # back off longer each time
            print(f"Rate-limited, waiting {wait}s (attempt {attempt+1}/{max_retries})")
            time.sleep(wait)
    raise RuntimeError("Still rate-limited after retries")

In [ ]:
def normalize(out):
    out = out.lower()
    for lab in labels:
        if lab.lower() in out:
            return lab
    return "Labour"   

In [10]:
# Limiting rates as I'm using free openrouter access

preds = []
for s in X_test:
    preds.append(normalize(classify(ZERO_SHOT_PROMPT.format(labels=label_str, speech=s[:4000]))))
    time.sleep(3.5)                                                      # roughly 17 req/min, less tahn the 20/min cap

NameError: name 'X_test' is not defined

In [17]:
print(classify("Reply with one word: hello"))

Rate-limited, waiting 30s (attempt 1/5)
Rate-limited, waiting 60s (attempt 2/5)
Rate-limited, waiting 90s (attempt 3/5)
Rate-limited, waiting 120s (attempt 4/5)
Rate-limited, waiting 150s (attempt 5/5)


RuntimeError: Still rate-limited after retries